# 02 · Runtime (Saanvi)
Baseline → ridge → gradient-boosted trees → quantile range with a conformal correction. Tuning on walk-forward folds only; test scored once.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = ROOT / "results"
load = lambda n: json.loads((R / n).read_text())
pd.set_option("display.precision", 4)

## Walk-forward results (development data only)

In [2]:
cv = load('runtime_cv.json'); pd.DataFrame([{'model': r['label'], 'mean MAE': r['mean_mae'], 'mean RMSE': r['mean_rmse']} for r in cv['ridge'] + cv['tree']])

,model,mean MAE,mean RMSE
0,ridge alpha=0.1,0.3523,0.5360
1,ridge alpha=10,0.3521,0.5357
2,ridge alpha=1000,0.3598,0.5373
3,"hgb {'loss': 'squared_error', 'learning_rate':...",0.3186,0.5023
4,"hgb {'loss': 'squared_error', 'learning_rate':...",0.3158,0.5011
5,"hgb {'loss': 'squared_error', 'learning_rate':...",0.3168,0.4999
6,"hgb {'loss': 'squared_error', 'learning_rate':...",0.3171,0.5026
7,"hgb {'loss': 'absolute_error', 'learning_rate'...",0.3062,0.5184
8,"hgb {'loss': 'absolute_error', 'learning_rate'...",0.3001,0.5113
9,"hgb {'loss': 'absolute_error', 'learning_rate'...",0.3031,0.5131


In [3]:
pd.Series(load('runtime_config.json'))

ridge_alpha                                                                   10
tree_params                    {'loss': 'absolute_error', 'learning_rate': 0....
max_iter                                                                     300
keep_tree                                                                   True
tree_beats_ridge_every_fold                                                 True
ridge_mean_mae                                                            0.3521
tree_mean_mae                                                             0.2995
quantile_mean_coverage                                                    0.7881
use_cqr                                                                     True
cqr_offset_log10                                                          0.1833
dtype: object

## Interval coverage: raw vs conformal (folds)

In [4]:
pd.DataFrame(load('runtime_intervals_cv.json')['folds'])

,fold,offset_from_fold,offset_log10,coverage_raw,width_raw,coverage_cqr,width_cqr
0,2,1,0.1546,0.8034,0.9419,0.9075,1.2512
1,3,2,0.1297,0.7848,1.0548,0.8818,1.3143


## Final test (run once)

In [5]:
F = load('final_metrics.json'); o = F['runtime']['overall']
pd.DataFrame({'model': o['model'], 'baseline': o['baseline']})

,model,baseline
n,171672.0000,171672.0000
mae,0.3382,0.6012
rmse,0.5448,0.8161
mean_signed_error,0.0553,0.1172
median_abs_error,0.1765,0.4226
error_factor,2.1786,3.9922
coverage_90,0.8860,NaN
interval_width_log10,1.3672,NaN


In [6]:
pd.DataFrame(o['ci'], index=['2.5%', '97.5%']).T

,2.5%,97.5%
mae_model,0.2596,0.4112
mae_base,0.4657,0.7079
mae_diff,-0.3238,-0.1758
rmse_diff,-0.3255,-0.2142


## Per band, seen/unseen, heavy user

In [7]:
pd.DataFrame({k: {'n': v['model']['n'], 'MAE model': v['model']['mae'], 'MAE baseline': v['baseline']['mae'], 'diff lo': v['ci']['mae_diff'][0], 'diff hi': v['ci']['mae_diff'][1], 'signed error': v['model']['mean_signed_error'], 'coverage': v['model'].get('coverage_90')} for k, v in F['runtime']['slices'].items()}).T

,n,MAE model,MAE baseline,diff lo,diff hi,signed error,coverage
band_1,43980.0,0.2980,0.5404,-0.3055,-0.1618,-0.0238,0.9237
band_2,42071.0,0.3568,0.6564,-0.3754,-0.1728,0.0977,0.8766
band_3,46679.0,0.3776,0.5924,-0.2635,-0.1382,0.0982,0.8600
band_4,38942.0,0.3162,0.6207,-0.3648,-0.1979,0.0473,0.8845
seen_user,169875.0,0.3381,0.6033,-0.3209,-0.1837,0.0568,0.8849
unseen_user,1797.0,0.3504,0.4018,-0.1702,-0.0116,-0.0880,0.9844
has_history,171579.0,0.3379,0.6010,-0.3219,-0.1792,0.0552,0.8859
no_history,93.0,0.8704,0.9673,-0.1828,-0.0016,0.1920,0.9677
heavy_user_excluded,126223.0,0.2787,0.5141,-0.3190,-0.1589,0.0446,0.9437
heavy_user_only,45449.0,0.5034,0.8432,-0.3398,-0.3398,0.0849,0.7255


## Ablation ladder

In [8]:
pd.DataFrame([{'rung': a['rung'], 'MAE': a['model']['mae'], 'diff lo': a['ci']['mae_diff'][0], 'diff hi': a['ci']['mae_diff'][1]} for a in F['ablation']['runtime']])

,rung,MAE,diff lo,diff hi
0,resources,0.6220,-0.0327,0.0783
1,+ task type,0.6069,-0.0478,0.0660
2,+ history,0.3524,-0.3170,-0.1552
3,+ time,0.3382,-0.3284,-0.1779


## Representative errors

In [9]:
e = load('errors.json'); pd.DataFrame([c for c in e['cases'] if c['kind'] == 'runtime'])

,kind,rule,y,tree,q05,q95,baseline,band,role_set,main_role,...,submit_hour,submit_weekday,user_n_finished,user_median_logrt,user_fail_rate,group_n_finished,group_median_logrt,group_fail_rate,user_submits_last_1h,case
0,runtime,largest over-prediction,0.8451,4.5739,2.3344,5.1431,4.7002,4,PyTorchWorker,PyTorchWorker,...,0,5,265.0,4.7002,0.3396,0.0,NaN,NaN,35.0,1
1,runtime,largest over-prediction,0.8451,4.5699,2.3166,5.1620,4.6920,4,PyTorchWorker,PyTorchWorker,...,0,5,271.0,4.6920,0.3321,0.0,NaN,NaN,39.0,2
2,runtime,largest under-prediction,5.4000,2.0655,1.2460,4.0045,1.8351,3,tensorflow,tensorflow,...,11,2,77.0,1.8351,0.8701,0.0,NaN,NaN,8.0,3
3,runtime,largest under-prediction,4.7810,1.4642,1.1932,2.5420,3.3199,2,PyTorchWorker,PyTorchWorker,...,10,1,247690.0,3.3199,0.4611,3.0,1.2553,0.0000,43.0,4
4,runtime,typical miss (|error| near the median),2.4814,2.6578,1.9454,3.8120,2.5179,4,tensorflow,tensorflow,...,10,6,331.0,2.5179,0.0272,2.0,2.7076,0.0000,4.0,5
5,runtime,typical miss (|error| near the median),3.5813,3.7579,3.3801,4.1395,3.9078,3,tensorflow,tensorflow,...,23,3,27297.0,3.9078,0.1185,52.0,3.6593,0.0385,0.0,6
6,runtime,user with no finished history,1.7076,2.8919,1.1326,4.6201,2.8357,2,tensorflow,tensorflow,...,15,0,0.0,NaN,NaN,0.0,NaN,NaN,2.0,7
7,runtime,"explicit GPU type requested (rare), largest miss",1.8573,4.8466,2.2753,5.3456,4.8411,3,PyTorchWorker,PyTorchWorker,...,15,3,121.0,4.8411,0.8926,0.0,NaN,NaN,2.0,8
8,runtime,baseline much better than model,4.8791,2.2772,2.0173,4.3798,3.3789,1,tensorflow,tensorflow,...,0,4,153232.0,3.3789,0.2218,57.0,4.1853,0.7193,768.0,9


In [10]:
pd.DataFrame(e['summary']['runtime_mae_by_true_length'])

,bin,n,mae_model,mse_model,mae_base
0,<10s,347,0.3824,0.3824,1.1621
1,10s-100s,39916,0.4615,0.4304,0.8903
2,100s-17m,60701,0.2717,0.1109,0.4511
3,17m-2.8h,51784,0.2937,-0.1433,0.4131
4,2.8h-28h,18111,0.3801,-0.3432,0.9344
5,>28h,813,1.1283,-1.1266,1.9384
